# Hadith Summarization - Colab Inference

Notebook ini menjalankan komputasi berat untuk abstractive summarization di Google Colab. Notebook lama `bukhari_ts (3).ipynb` tetap read-only dan tidak digunakan sebagai file kerja.

## 1. Instal dependency

Jalankan cell ini di Colab runtime. Di laptop lokal, gunakan `requirements-local.txt`.

In [ ]:
!pip install -q transformers sentencepiece accelerate evaluate rouge-score bert-score openpyxl

## 2. Upload dataset

Upload `dataset_hadis.csv` dengan kolom default: `Num_hadith`, `teks_arab`, dan `terjemahan`. Jika nama kolom berbeda, ubah variabel konfigurasi pada cell berikut.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_DATA_PATH = next(iter(uploaded.keys()))

In [ ]:
from pathlib import Path

HADITH_ID_COLUMN = "Num_hadith"
ARABIC_TEXT_COLUMN = "teks_arab"
INDONESIAN_TEXT_COLUMN = "terjemahan"
REFERENCE_SUMMARY_COLUMN = "ringkasan_referensi"

ABSTRACTIVE_MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"
MAX_INPUT_LENGTH = 512
MAX_SUMMARY_LENGTH = 128
MIN_SUMMARY_LENGTH = 20
NUM_BEAMS = 4
NO_REPEAT_NGRAM_SIZE = 3
BATCH_SIZE = 4
LIMIT_ROWS = None

OUTPUT_DATA_PATH = Path("hasil_ringkasan_hadis_colab.csv")

## 3. Load dan validasi dataset

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup

def read_dataset(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, encoding="utf-8")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Format dataset tidak didukung: {suffix}")

def validate_columns(df, required_columns):
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Kolom wajib tidak ditemukan: {missing}. Kolom tersedia: {list(df.columns)}")

CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
WHITESPACE_RE = re.compile(r"\s+")

def clean_indonesian_text(text):
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = CONTROL_CHARS_RE.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()

df = read_dataset(INPUT_DATA_PATH)
validate_columns(df, [HADITH_ID_COLUMN, ARABIC_TEXT_COLUMN, INDONESIAN_TEXT_COLUMN])
df = df.fillna("")
df = df[df[INDONESIAN_TEXT_COLUMN].astype(str).str.strip() != ""].copy()
if LIMIT_ROWS:
    df = df.head(LIMIT_ROWS).copy()
df["teks_sumber_ringkasan"] = df[INDONESIAN_TEXT_COLUMN].map(clean_indonesian_text)
print(df.shape)
df.head()

## 4. Jalankan abstractive summarization

In [ ]:
import torch
import re
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def word_count(text):
    return len(re.findall(r"\S+", str(text).strip()))

def compression_ratio(summary, source):
    source_words = word_count(source)
    if source_words == 0:
        return 0.0
    return round(word_count(summary) / source_words, 4)

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(ABSTRACTIVE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(ABSTRACTIVE_MODEL_NAME).to(device)
model.eval()
print("Device:", device)

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")

def split_sentences(text):
    sentences = [sentence.strip() for sentence in SENTENCE_SPLIT_RE.split(str(text)) if sentence.strip()]
    return sentences or [str(text).strip()]

def needs_chunking(text):
    encoded = tokenizer(
        text,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )
    return len(encoded["input_ids"]) > MAX_INPUT_LENGTH

def chunk_by_sentence(text):
    chunks = []
    current = ""
    for sentence in split_sentences(text):
        candidate = f"{current} {sentence}".strip()
        if current and needs_chunking(candidate):
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks or [text]

def generate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            num_beams=NUM_BEAMS,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            early_stopping=True,
            max_length=MAX_SUMMARY_LENGTH,
            min_length=MIN_SUMMARY_LENGTH,
        )
    return [tokenizer.decode(output, skip_special_tokens=True).strip() for output in outputs]

def summarize_long_text(text):
    chunks = chunk_by_sentence(text)
    partial_summaries = []
    for start in range(0, len(chunks), BATCH_SIZE):
        partial_summaries.extend(generate_batch(chunks[start:start + BATCH_SIZE]))
    combined = " ".join(partial_summaries).strip()
    if combined and needs_chunking(combined):
        return summarize_long_text(combined)
    if combined:
        return generate_batch([combined])[0]
    return ""

texts = df["teks_sumber_ringkasan"].astype(str).tolist()
summaries = [""] * len(texts)
errors = [""] * len(texts)
used_chunking_values = [False] * len(texts)
pending = []

for index, text in enumerate(texts):
    source = text.strip() if isinstance(text, str) else ""
    if not source:
        errors[index] = "empty_input"
    elif needs_chunking(source):
        try:
            summaries[index] = summarize_long_text(source)
            used_chunking_values[index] = True
        except Exception as exc:
            errors[index] = str(exc)
    else:
        pending.append((index, source))

for start in tqdm(range(0, len(pending), BATCH_SIZE)):
    batch = pending[start:start + BATCH_SIZE]
    batch_texts = [item[1] for item in batch]
    try:
        batch_summaries = generate_batch(batch_texts)
        for (index, _), summary in zip(batch, batch_summaries):
            summaries[index] = summary
    except Exception as exc:
        for index, _ in batch:
            errors[index] = str(exc)

df["ringkasan_abstractive"] = summaries
df["status_abstractive"] = ["ok" if summary else "error" for summary in summaries]
df["model_abstractive"] = ABSTRACTIVE_MODEL_NAME
df["jumlah_kata_abstractive"] = df["ringkasan_abstractive"].map(word_count)
df["compression_ratio_abstractive"] = df.apply(lambda row: compression_ratio(row["ringkasan_abstractive"], row["teks_sumber_ringkasan"]), axis=1)
df["used_chunking"] = used_chunking_values
df["error_abstractive"] = errors
df[[HADITH_ID_COLUMN, "ringkasan_abstractive", "status_abstractive"]].head()

## 5. Simpan dan download output

Pindahkan file ini ke `hadith_summarization/data/output/hasil_ringkasan_hadis_colab.csv` di laptop, lalu jalankan `python run_light_pipeline.py` untuk menggabungkannya dengan hasil lokal.

In [ ]:
output_columns = [
    HADITH_ID_COLUMN,
    "ringkasan_abstractive",
    "status_abstractive",
    "model_abstractive",
    "jumlah_kata_abstractive",
    "compression_ratio_abstractive",
    "used_chunking",
    "error_abstractive",
]
df[output_columns].to_csv(OUTPUT_DATA_PATH, index=False, encoding="utf-8-sig")
files.download(str(OUTPUT_DATA_PATH))